In [ ]:
#| default_exp build

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import ast, math, re, sys
from functools import lru_cache
import numpy as np
from fastcore.all import AttrDict, L, Path, chunked, defaults, first, ifnone, merge, patch, store_attr
from fastcore.parallel import ProcessPoolExecutor
from fastlite import Database
from apswutils.db import Table
from multiprocessing import get_context
from litesearch.core import (_in, _rid, _slug, _np_dtype, process_content, write_txn, db_lock,
                             upsert_all, rrf_all)
from litesearch.topics import get_graph
from litesearch.utils import hash_embed
from vruksha.entities import (_acr, _jac, _lex_ok, _norm, _nums, _sentences, _toks, _yake_terms,
                              code_entities, prose_windows, text_entities)

## Building the graph

In [ ]:
#| export
_CODE_TYPES = {'FunctionDef','AsyncFunctionDef','ClassDef'}

def _is_code(chunk):
    md = chunk.get('metadata') or {}
    if not isinstance(md, dict): return False
    return md.get('lang') == '.py' or md.get('type') in _CODE_TYPES

def _pmi_edges(wins,              # list of entity-id sets (one per co-occurrence window)
               min_n=2,           # min co-occurrence count
               min_npmi=0.15,     # min normalized PMI
               max_df=0.4,        # drop entities present in more than this fraction of windows
               max_degree=48,     # keep only the strongest edges per node
               rel='cooc'):
    'Normalized-PMI co-occurrence edges over windows. max_df is what kills hub terms like "section".'
    n = len(wins)
    if n < 2: return []
    if hasattr(wins, 'pair_counts'):                 # a _WindowStore counts on disk, same numbers
        cnt = wins.entity_counts()
        hub = {e for e, c in cnt.items() if c / n > max_df}
        pairs = ((a, b, c) for a, b, c in wins.pair_counts(hub, min_n))
    else:
        cnt = {}
        for w in wins:
            for e in w: cnt[e] = cnt.get(e, 0) + 1
        hub = {e for e, c in cnt.items() if c / n > max_df}
        pair = {}
        for w in wins:
            es = sorted(e for e in w if e not in hub)
            for i, a in enumerate(es):
                for b in es[i+1:]: pair[(a, b)] = pair.get((a, b), 0) + 1
        pairs = ((a, b, c) for (a, b), c in pair.items())
    out = []
    for a, b, c in pairs:
        if c < min_n: continue
        pa, pb, pab = cnt[a]/n, cnt[b]/n, c/n
        npmi = math.log(pab/(pa*pb)) / (-math.log(pab)) if 0 < pab < 1 else 0.0
        if npmi < min_npmi: continue
        out.append(dict(src=a, dst=b, rel=rel, weight=round(npmi, 5), n=c))
    if max_degree:
        # (src, dst) breaks weight ties explicitly. Without it the survivors of the degree cap depend
        # on the order pairs were counted in, which differs between the in-memory dict and SQLite's
        # group-by — 30 of 2,707 edges on a 500-chunk corpus, all of them ties, none of them wrong.
        deg, kept = {}, []
        for e in sorted(out, key=lambda r: (-r['weight'], r['src'], r['dst'])):
            if deg.get(e['src'], 0) < max_degree and deg.get(e['dst'], 0) < max_degree:
                deg[e['src']] = deg.get(e['src'], 0)+1; deg[e['dst']] = deg.get(e['dst'], 0)+1
                kept.append(e)
        out = kept
    return out

# Below this many prose chunks in one drain, a process pool costs more to start than it saves.
MIN_PARALLEL_CHUNKS = 200

def _prose_job(txt):
    '''Windows for one prose chunk, as plain tuples. Module level so a process pool can pickle it.

    Only text crosses the boundary and only tuples come back — no database handle, nothing that a
    fork would have to keep consistent.'''
    return [[(s, k) for s, k in w] for w in prose_windows(txt)]

def _pool(nw):
    '''A `ProcessPoolExecutor` started the way `fastcore.parallel` starts one.

    fork, not spawn, on darwin. A spawned worker unpickles the job by module and name, and under
    nbdev `_prose_job` is defined in the notebook's `__main__`, which the worker has no copy of —
    so every worker died on unpickle. Only text crosses the boundary and only tuples come back,
    so there is nothing here a fork has to keep consistent.'''
    kw = dict(mp_context=get_context('fork')) if sys.platform == 'darwin' else {}
    return ProcessPoolExecutor(nw, **kw)

def _n_workers(n, n_workers):
    'Resolve `n_workers`: None picks by size, 0 stays serial, anything else is taken literally.'
    if n_workers is not None: return n_workers
    return 0 if n < MIN_PARALLEL_CHUNKS else defaults.cpus

`_pmi_edges` walks its windows twice, because the hub cut-off is not known until entities are
counted. `_WindowStore` keeps them in SQLite so the second pass does not hold a corpus in memory.

In [ ]:
#| export
class _WindowStore:
    '''Co-occurrence windows kept in a SQLite table instead of a python list.
    The table is TEMP: scratch for one build does not belong in the shared schema, where creating
    and dropping it bumps the schema cookie and forces every other connection to re-prepare.'''
    PAGE = 10_000   # windows per read; keeps the cursor closed between yields
    def __init__(self, db, name):
        store_attr(); self.n, self._w = 0, 0
        with write_txn(db):
            db.conn.execute(f'drop table if exists main.{name}')   # left behind by older versions
            db.conn.execute(f'create temp table if not exists {name} (w integer, e text)')
            db.conn.execute(f'delete from {name}')
    def extend(self, wins):
        rows = [(self._w + i, e) for i, w in enumerate(wins) for e in w if e]
        if not rows: self._w += len(wins); self.n += len(wins); return
        with write_txn(self.db): self.db.conn.executemany(f'insert into {self.name} values (?,?)', rows)
        self._w += len(wins); self.n += len(wins)
    def entity_counts(self):
        'entity -> windows containing it. Bounded by vocabulary, which saturates; safe in memory.'
        with db_lock(self.db):
            return dict(self.db.conn.execute(f'select e, count(*) from {self.name} group by e'))
    def pair_counts(self, hub, min_n):
        """Co-occurrence count per unordered pair, aggregated on disk. Materialised: the caller
        builds edges between rows, and an open cursor makes the connection busy for that whole walk."""
        c = self.db.conn
        with write_txn(self.db):
            c.execute(f'create index if not exists {self.name}_w on {self.name}(w)')
            c.execute(f'create temp table if not exists {self.name}_hub (e text primary key)')
            c.execute(f'delete from {self.name}_hub')
            if hub: c.executemany(f'insert or ignore into {self.name}_hub values (?)', [(e,) for e in hub])
        with db_lock(self.db): return c.execute(f'''select a.e, b.e, count(*) c from {self.name} a
              join {self.name} b on a.w = b.w and a.e < b.e
              where a.e not in (select e from {self.name}_hub)
                and b.e not in (select e from {self.name}_hub)
              group by a.e, b.e having c >= {int(min_n)}''').fetchall()
    def __len__(self): return self.n
    def __iter__(self):
        'One page of whole windows at a time: a cursor left open across a yield holds the connection.'
        last = -1
        while True:
            with db_lock(self.db):
                ws = [r[0] for r in self.db.conn.execute(
                    f'select distinct w from {self.name} where w > ? order by w limit {self.PAGE}', (last,)).fetchall()]
                if not ws: return
                rows = self.db.conn.execute(f'select w, e from {self.name} where w between ? and ? order by w',
                                            (ws[0], ws[-1])).fetchall()
            w, acc = None, set()
            for wi, e in rows:
                if wi != w:
                    if w is not None: yield acc
                    w, acc = wi, set()
                acc.add(e)
            if w is not None: yield acc
            last = ws[-1]
    def drop(self):
        with write_txn(self.db):
            self.db.conn.execute(f'drop table if exists temp.{self.name}')
            self.db.conn.execute(f'drop table if exists temp.{self.name}_hub')

def build_graph(db,                  # Database with a chunk store
                chunks,              # chunk dicts ({'content','metadata'}) as returned by dir2chunks/pkg2chunks
                store='store',       # chunk store name
                prefix=None,         # graph table prefix
                terms_fn=None,       # (text, topk) -> terms, replacing yake (see `sanskrit_terms`)
                emb_fn=None,         # embedder for entity names (required for resolve_entities)
                code=True,           # extract AST symbols from code chunks
                prose=True,          # extract surfaces from prose chunks
                cooc=True,           # also write PMI co-occurrence edges over sentence windows
                min_n=2,             # min co-occurrence count
                min_npmi=0.15,       # min normalized PMI
                max_df=0.4,          # drop entities present in >max_df of windows
                max_degree=48,       # max cooc edges kept per node
                batch:int=None,      # chunks per flush; keeps windows on disk instead of in memory
                n_workers:int=None): # extraction workers; 0 is serial, None picks by queue size
    '''Extract entities + mentions + edges (exact for code, PMI co-occurrence for prose) from chunks.

    `batch` is what makes a large corpus finish. Left as None everything accumulates in memory for
    the whole call, which is fine for a package or a few thousand pages and is not fine for a
    corpus: windows are one per sentence and never stop arriving, so memory tracks the corpus with
    no bound. Set it and mentions are flushed every `batch` chunks and windows go to a scratch
    table, which `_pmi_edges` reads back twice exactly as it would a list. The edges are identical
    either way — the batched path is a different place to keep the same numbers.

    `n_workers` spreads extraction across processes, which is where a build spends 88% of its time.
    Use it with `batch` large enough to be worth a pool: the queue is drained once per batch, so a
    small `batch` pays pool startup repeatedly for very little work.'''
    g = db.get_graph(store, prefix)
    ents, mens, edges = {}, {}, {}
    wins = _WindowStore(db, f'{g.prefix}_win_scratch') if batch else []
    def ent(name, kind):
        'Register an entity by canonical name, returning its hash id.'
        n = _norm(name)
        if not n: return None
        i = _slug(n)
        e = ents.setdefault(i, dict(content=n, kind=kind, freq=0, canon=i))
        e['freq'] += 1
        return i
    def men(cid, eid, surface):
        if not (cid and eid): return
        m = mens.setdefault((cid, eid), dict(chunk_id=cid, entity_id=eid, surface=surface, n=0))
        m['n'] += 1
    def edge(s, d, rel, w=1.0):
        if not (s and d) or s == d: return
        e = edges.setdefault((s, d, rel), dict(src=s, dst=d, rel=rel, weight=0.0, n=0))
        e['weight'] += w; e['n'] += 1
    n_mens = 0
    def flush_mentions():
        'Mentions are complete once their chunk is processed, so they need not wait for the corpus.'
        nonlocal n_mens
        if not mens: return
        upsert_all(g.mentions, mens.values(), ('chunk_id','entity_id'))
        n_mens += len(mens); mens.clear()

    pend = []                                   # windows waiting for the next flush
    def add_wins(ws):
        if batch: pend.extend(ws)
        else: wins.extend(ws)

    prose_q, pool = [], None
    def prose_wins():
        '''Windows per queued prose chunk, over a process pool once there are enough to pay for one.
        Order is preserved by both pools, and the reduce depends on it: `ents.setdefault` keeps the
        `kind` of an entity's *first* mention, so a reordered stream would relabel entities.'''
        nonlocal pool
        nw = 0 if terms_fn is not None else _n_workers(len(prose_q), n_workers)
        if nw and nw > 1:
            if pool is None: pool = _pool(nw)
            # both sides materialised: `drain_prose` clears the queue these are drawn from
            cids, txts = [c for c, _ in prose_q], [t for _, t in prose_q]
            return zip(cids, pool.map(_prose_job, txts))
        return ((cid, prose_windows(txt, terms_fn=terms_fn)) for cid, txt in prose_q)

    def drain_prose():
        'Turn the queued prose chunks into entities, mentions and windows, then clear the queue.'
        for cid, wl in prose_wins():
            for win in wl:
                w = set()
                for surf, kind in win:
                    i = ent(surf, kind)
                    if i: men(cid, i, surf); w.add(i)
                if len(w) > 1: add_wins([w])
        prose_q.clear()

    n_seen = 0
    try:
        for c in ([chunks] if isinstance(chunks, dict) else chunks):
            txt = c.get('content')
            if not (txt and txt.strip()): continue
            cid = c.get('id') or _slug(txt)
            if code and _is_code(c):
                dname, calls, imps = code_entities(c)
                did = ent(dname, 'symbol') if dname else None
                if did: men(cid, did, dname)
                w = {did} if did else set()
                for nm in calls:
                    i = ent(nm, 'symbol')
                    men(cid, i, nm)
                    edge(did, i, 'calls')
                    w.add(i)
                for nm in imps:
                    i = ent(nm, 'module')
                    men(cid, i, nm)
                    edge(did, i, 'imports')
                    w.add(i)
                if len(w) > 1: add_wins([w - {None}])
            elif prose: prose_q.append((cid, txt))
            n_seen += 1
            if batch and n_seen % batch == 0:
                drain_prose()
                wins.extend(pend)
                pend.clear()
                flush_mentions()
        drain_prose()
    finally:
        if pool is not None: pool.shutdown()
    if batch: wins.extend(pend); pend.clear()
    rows = list(ents.values())
    if cooc and len(wins):
        for e in _pmi_edges(wins, min_n, min_npmi, max_df, max_degree): edges[(e['src'], e['dst'], e['rel'])] = e
    n_wins = len(wins)
    if batch: wins.drop()
    with write_txn(db):
        if rows:
            if emb_fn: process_content(g.entities, rows, embed=True, emb_fn=emb_fn)
            else: g.entities.insert_all(rows, upsert=True, hash_id='id', hash_id_columns=['content'])
        if mens:
            upsert_all(g.mentions, mens.values(), ('chunk_id','entity_id')); n_mens += len(mens)
        if edges: upsert_all(g.edges, edges.values(), ('src','dst','rel'))
    if emb_fn and rows: g.entities.rebuild_index()
    return dict(entities=len(rows), mentions=n_mens, edges=len(edges), windows=n_wins)


In [ ]:
#| hide
# `batch=` moves where the windows live, not what they contain: entities, mentions and edges all
# have to come back identical to the in-memory path, including the degree-capped edge set.
import hashlib
from fastcore.test import test_eq
from litesearch.core import database
_gtexts = ['polonium and radium were isolated from pitchblende residue by marie curie.',
           'radium salts glow; polonium decays fast. pitchblende is the ore.',
           'uranium ore yields radium. marie curie measured the decay of polonium.',
           'thorium series decay differs from the uranium series decay chain.',
           'pitchblende residue contains uranium, radium and polonium together.']
def _gbuild(batch):
    db = database()
    db.get_store('store', hash=True, ann=True)
    st = build_graph(db, [dict(content=f'[{i}] {t}') for i in range(6) for t in _gtexts],
                     emb_fn=lambda ts, **kw: hash_embed(ts, ndim=64), batch=batch)
    dump = lambda rows: hashlib.sha256(repr(sorted(map(tuple, rows))).encode()).hexdigest()[:12]
    return st, (dump([(r['id'], r['content'], r['freq']) for r in db.t.entities(select='id,content,freq')]),
                dump([(r['chunk_id'], r['entity_id'], r['n']) for r in db.t.mentions(select='chunk_id,entity_id,n')]),
                dump([(r['src'], r['dst'], r['rel'], round(r['weight'], 5), r['n']) for r in db.t.edges()]))
_sa, _ha = _gbuild(None)
_sb, _hb = _gbuild(4)
test_eq(_ha, _hb)
test_eq(_sa, _sb)

# and a generator is not materialised on the way in — `batch` plus a generator is what keeps a
# corpus larger than memory off the heap
_gen = (dict(content=f'[{i}] {t}') for i in range(6) for t in _gtexts)
_dbg = database(); _dbg.get_store('store', hash=True, ann=True)
test_eq(build_graph(_dbg, _gen, emb_fn=lambda ts, **kw: hash_embed(ts, ndim=64), batch=4), _sa)

# `mentions` counts what this build wrote, the way `entities` and `edges` already did
_mdb = database(); _mdb.get_store('store', hash=True, ann=True)
_mchunks = [dict(content=f'[{i}] {t}') for i in range(6) for t in _gtexts]
_menc = lambda ts, **kw: hash_embed(ts, ndim=64)
_m1 = build_graph(_mdb, _mchunks, emb_fn=_menc)
_mdb.t.mentions.insert(dict(chunk_id='elsewhere', entity_id='elsewhere', surface='x', n=1))
_m2 = build_graph(_mdb, _mchunks, emb_fn=_menc)
test_eq(_m1['mentions'], _m2['mentions'])
test_eq(_mdb.t.mentions.count, _m2['mentions'] + 1)      # the foreign row is still there, just not counted


In [ ]:
#| hide
# Extraction on a pool must reduce to the same graph as extraction in the parent. The reduce depends
# on order — `ents.setdefault` keeps the `kind` of an entity's first mention — so this would fail if
# the pool ever returned results out of order.
_ptexts = ['polonium and radium were isolated from pitchblende residue by marie curie.',
           'radium salts glow in the dark; polonium decays fast. pitchblende is the ore.',
           'uranium ore yields radium. marie curie measured the decay of polonium carefully.',
           'the thorium series decay differs from the uranium series decay chain entirely.']
def _pbuild(nw):
    db = database(); db.get_store('store', hash=True, ann=True)
    st = build_graph(db, [dict(content=f'[{i}] {t}') for i in range(12) for t in _ptexts],
                     emb_fn=lambda ts, **kw: hash_embed(ts, ndim=64), n_workers=nw)
    dump = lambda rows: hashlib.sha256(repr(sorted(map(tuple, rows))).encode()).hexdigest()[:12]
    return st, (dump([(r['id'], r['content'], r['kind'], r['freq']) for r in db.t.entities(select='id,content,kind,freq')]),
                dump([(r['src'], r['dst'], r['rel'], round(r['weight'], 5), r['n']) for r in db.t.edges()]))
_s0, _h0 = _pbuild(0)          # serial
_s2, _h2 = _pbuild(2)          # forced onto a pool despite being under MIN_PARALLEL_CHUNKS
test_eq(_h0, _h2)
test_eq(_s0, _s2)


In [ ]:
#| hide
# A `terms_fn` is a closure over resources that do not pickle, so extraction has to stay serial
# when one is given. The pool is the default path on a corpus, so a `terms_fn` that travelled no
# further than the parent process would leave the workers building the graph out of yake output.
from litesearch.core import database
_calls = []
def _terms(text, topk=12):
    _calls.append(text)
    return L(['polonium', 'marie curie'])

_pb = [dict(content='Marie Curie studied polonium.', metadata=dict(lang='.txt')),
       dict(content='Polonium decays by alpha emission.', metadata=dict(lang='.txt'))]
_r = build_graph(database(), _pb, code=False, terms_fn=_terms, n_workers=4)
assert len(_calls) == len(_pb), _calls        # ran in this process, for every chunk
assert _r['entities'] == 2 and _r['mentions'] == 3 and _r['windows'] == 1, _r


## Entity resolution

`_uf_union` merges two groups only when the merged group stays a clique under `_lex_ok`. Without
that, containment chains transitively: `polonium` merges `isolated polonium` merges `polonium
decay`, and the graph collapses to one node.

In [ ]:
#| export
_EXACT_KINDS = ('symbol', 'module', 'topic')   # names that are already canonical — never merge these

def _uf_find(par, x):
    while par[x] != x: par[x] = par[par[x]]; x = par[x]
    return x

def _uf_union(par, rank, a, b, name=None, ok=None, members=None, max_check=32):
    'Merge two groups. With `name`/`ok`, only if the merged group stays a **clique** under `ok`.'
    ra, rb = _uf_find(par, a), _uf_find(par, b)
    if ra == rb: return False
    if name is not None and ok is not None:
        A = (members.get(ra, [ra]) if members else [ra])[:max_check]
        B = (members.get(rb, [rb]) if members else [rb])[:max_check]
        if not all(ok(name[x], name[y]) for x in A for y in B): return False
    if rank[ra] < rank[rb]: ra, rb = rb, ra
    par[rb] = ra
    if members is not None: members.setdefault(ra, [ra]).extend(members.pop(rb, [rb]))
    return True

def _ann_pairs(tbl, rows, k=8, dtype=np.float16):
    '''`(id, neighbour_id, distance)` for every embedded row, from one batched HNSW probe.

    usearch searches a matrix of queries across its own thread pool, so asking it once for 8,487
    vectors is not the same work as asking it 8,487 times: the per-entity loop also ran one
    `rowid IN (...)` query per entity, and the two together were 10.3s of a 23.0s resolve. The
    candidate set is unchanged — same k, same neighbours, same distances.'''
    idx_ = tbl.db.get_index(tbl.name)
    emb = [r for r in rows if r['embedding']]
    if not emb or not idx_.size: return
    key = {r['rowid']: r for r in tbl.db.q(f'select {_rid()}, id, content from {tbl.name}')}
    res = idx_.search(np.stack([np.frombuffer(r['embedding'], dtype=dtype) for r in emb]),
                      count=min(k, idx_.size))
    ks, ds = np.atleast_2d(res.keys), np.atleast_2d(res.distances)
    for r, kr, dr in zip(emb, ks, ds):
        for kk, dd in zip(np.atleast_1d(kr).tolist(), np.atleast_1d(dr).tolist()):
            if (o := key.get(int(kk))): yield r, o, float(dd)

def _lexical_pairs(name, max_group=60):
    '''Candidate merge pairs by shared-token blocking — catches containment variants

    Tokens are walked in sorted order, which is what makes a resolve reproducible. `_toks` returns
    a frozenset, so its iteration order follows string hashes and therefore `PYTHONHASHSEED`; that
    decided the insertion order of `inv`, which decided the order pairs were proposed in, and
    `_uf_union` only accepts a merge that keeps the group a clique — an order-dependent test. Two
    resolves of the *same* database in two processes came back with different partitions and merge
    counts drifting over a range of three, which looked like HNSW noise and was not: hold the seed
    still and both the old code and the new one are exactly reproducible, and `verify_ann_probe`
    says the ANN candidates were stable the whole time.'''
    inv = {}
    for i, s in name.items():
        for t in sorted(_toks(s)):
            if len(t) > 2: inv.setdefault(t, []).append(i)
    seen = set()
    for ids in inv.values():
        if len(ids) < 2: continue
        if len(ids) <= max_group: pairs = ((ids[a], ids[b]) for a in range(len(ids)) for b in range(a+1, len(ids)))
        else:
            srt = sorted(ids, key=lambda i: name[i])
            pairs = ((srt[a], srt[b]) for a in range(len(srt)) for b in range(a+1, min(a+max_group, len(srt))))
        for x, y in pairs:
            p = (x, y) if x < y else (y, x)
            if p not in seen: seen.add(p); yield p

def resolve_entities(db,                # Database
                     store='store',     # chunk store the graph belongs to
                     prefix=None,       # graph table prefix
                     thresh=0.18,       # max ANN distance for a merge candidate
                     lex=0.34,          # min token Jaccard for the lexical guard
                     k=8,               # ANN neighbours considered per entity
                     lexical=True,      # also block on shared tokens (works without embeddings)
                     max_group=60,      # skip token groups bigger than this in the lexical pass
                     skip_kinds=_EXACT_KINDS,  # kinds whose names are already canonical
                     dtype=np.float16):
    'Merge near-duplicate entities: ANN + shared-token candidates, both gated by the lexical guard.'
    g = db.get_graph(store, prefix)
    allr = L(g.entities(select=f'{_rid()}, id, content, freq, embedding, kind, canon'))
    rows = allr.filter(lambda r: r['kind'] not in set(skip_kinds or ()))
    if len(rows) < 2:
        return dict(merged=0, by_ann=0, by_lexical=0, edges=len(list(g.edges())),
                    entities=len(allr), resolvable=len(rows), canonical=len(allr))
    par  = {r['id']: r['id'] for r in rows}
    rank = {r['id']: (r['freq'] or 0) for r in rows}
    name = {r['id']: r['content'] for r in rows}
    ann_m = lex_m = 0
    guard, members = (lambda x, y: _lex_ok(x, y, lex)), {}
    for r, h, dist in _ann_pairs(g.entities, rows, k, dtype):
        oid = h.get('id')
        if not oid or oid == r['id'] or oid not in par: continue
        if dist > thresh: continue
        if not _lex_ok(r['content'], h['content'], lex): continue
        if _uf_union(par, rank, r['id'], oid, name, guard, members): ann_m += 1
    if lexical:
        for a, b in _lexical_pairs(name, max_group):
            if _lex_ok(name[a], name[b], lex) and _uf_union(par, rank, a, b, name, guard, members): lex_m += 1
    upd = [(_uf_find(par, i), i) for i in par]
    was = {r['id']: r['canon'] for r in rows}
    chg = [(c, i) for c, i in upd if c != was.get(i)]
    if chg:
        with write_txn(db):
            db.conn.cursor().executemany(f'update {g.entities.name} set canon=? where id=?', chg)
    canon = {i: c for c, i in upd}
    n_edges = _collapse_edges(db, g, canon)
    skipped = len(allr) - len(rows)
    return dict(merged=ann_m+lex_m, by_ann=ann_m, by_lexical=lex_m, edges=n_edges,
                entities=len(allr), resolvable=len(rows),
                canonical=len({c for c, _ in upd}) + skipped)

def _collapse_edges(db, g, canon):
    'Rewrite edge endpoints onto canonical ids. Traversal reads src/dst straight from the table,'
    rows = list(g.edges())
    if not rows: return 0
    agg = {}
    for r in rows:
        s, d = canon.get(r['src'], r['src']), canon.get(r['dst'], r['dst'])
        if s == d: continue
        if s > d and r['rel'] == 'cooc': s, d = d, s     # cooc is symmetric; keep one direction
        k = (s, d, r['rel'])
        a = agg.setdefault(k, dict(src=s, dst=d, rel=r['rel'], weight=0.0, n=0))
        a['weight'] = max(a['weight'], r['weight'] or 0.0); a['n'] += (r['n'] or 0)
    with write_txn(db):
        g.edges.delete_where()
        upsert_all(g.edges, agg.values(), ('src','dst','rel'))
    return len(agg)


In [ ]:
#| hide
# Containment merges are wanted (`polonium` really is `isolated polonium`) and their *transitive
# closure* is not: the yake path emits overlapping sub-phrases, every adjacent pair passes the
# guard, and plain union-find walked the ladder from `marie curie` to `polonium` — measured 54
# entities collapsing into one 18-member group. Every merged group must be a clique under `_lex_ok`.
_par  = {c: c for c in 'abcd'}
_rank = {c: 1 for c in 'abcd'}
_nm   = {'a':'marie curie isolated','b':'curie isolated polonium','c':'isolated polonium','d':'polonium'}
_ok   = lambda x, y: _lex_ok(x, y)
_mem  = {}
_rungs = [('a','b'),('b','c'),('c','d')]
for _x, _y in _rungs: assert _ok(_nm[_x], _nm[_y]), (_x, _y)      # each rung is legitimate
assert not _ok(_nm['a'], _nm['d'])                                # the endpoints are not
for _x, _y in _rungs: _uf_union(_par, _rank, _x, _y, _nm, _ok, _mem)
assert _uf_find(_par,'a') != _uf_find(_par,'d'), 'the ladder was walked end to end'
# and every finished group really is a clique
for _root in {_uf_find(_par, c) for c in _par}:
    _g = [c for c in _par if _uf_find(_par, c) == _root]
    assert all(_ok(_nm[x], _nm[y]) for x in _g for y in _g), _g
# without the guard the same rungs collapse everything, which is the bug this pins
_p2, _r2 = {c: c for c in 'abcd'}, {c: 1 for c in 'abcd'}
for _x, _y in _rungs: _uf_union(_p2, _r2, _x, _y)
assert _uf_find(_p2,'a') == _uf_find(_p2,'d')


In [ ]:
#| hide
# The batched ANN probe has to pick the same candidates the per-entity loop did: same k, same
# neighbours, same distances. Compared here against the unbatched search it replaced.
from fastcore.test import test_eq
_rdb = database()
_rst = _rdb.get_store('store', hash=True, ann=True)
_rtexts = ['isolated polonium', 'polonium', 'radium salts', 'radium', 'pitchblende residue',
           'marie curie', 'curie', 'uranium ore', 'uranium', 'thorium series']
build_graph(_rdb, [dict(content=f'A passage discussing {t} in some detail.') for t in _rtexts],
            emb_fn=lambda ts, **kw: hash_embed(ts, ndim=64))
_ents = L(_rdb.t.entities(select=f'{_rid()}, id, content, freq, embedding, kind'))
_batched = {(r['id'], h['id']) for r, h, d in _ann_pairs(_rdb.t.entities, _ents, 8) if d <= 1.0}
_looped = {(r['id'], h['id']) for r in _ents if r['embedding']
           for h in _rdb.t.entities.ann_search(r['embedding'], columns=['id','content'], limit=8)
           if (h.get('_dist') or 1.0) <= 1.0}
test_eq(_batched, _looped)


In [ ]:
#| hide
# An oversized token block must still propose pairs. The regression this guards is silent: before,
# a block over `max_group` yielded nothing at all, so merges stopped being proposed precisely as a
# corpus grew large enough to need them.
_many = {f'id{i:03d}': f'radium variant {i:03d}' for i in range(90)}      # one block of 90 on 'radium'
_pairs = list(_lexical_pairs(_many, max_group=60))
test_eq(len(_pairs) > 0, True)
test_eq(all(a in _many and b in _many for a, b in _pairs), True)
test_eq(len(set(_pairs)), len(_pairs))                                    # no duplicate pairs
# a block within the cap is still exhaustive
_few = {f'id{i}': f'polonium variant {i}' for i in range(5)}
test_eq(len(list(_lexical_pairs(_few, max_group=60))), 5*4//2)


### `cooccur_edges`

Rebuilds edges from stored mentions with the whole chunk as the window. Run it after
`resolve_entities`, when the entity ids have settled.

In [ ]:
#| export
def cooccur_edges(db,                 # Database
                  store='store',      # chunk store
                  prefix=None,        # graph table prefix
                  min_n=2,            # min co-occurrence count
                  min_npmi=0.15,      # min normalized PMI (prunes stopword-ish hub nodes)
                  max_df=0.4,         # drop entities present in >max_df of chunks
                  max_degree=48,      # keep only the strongest edges per node
                  rel='cooc',
                  use_canon=True):    # collapse to canonical ids from resolve_entities
    'Rebuild co-occurrence edges from the stored mentions, using the chunk as the window.'
    g = db.get_graph(store, prefix)
    canon = {}
    if use_canon:
        canon = {r['id']: (r['canon'] or r['id']) for r in g.entities(select='id, canon')}
    cid_ents = {}
    for m in g.mentions(select='chunk_id, entity_id'):
        e = canon.get(m['entity_id'], m['entity_id'])
        cid_ents.setdefault(m['chunk_id'], set()).add(e)
    out = _pmi_edges(list(cid_ents.values()), min_n, min_npmi, max_df, max_degree, rel)
    if out: upsert_all(g.edges, out, ('src','dst','rel'))
    return len(out)


In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()